In [1]:
import os
import sys
import copy
import pandas as pd
import numpy as np

# get the absolute path of the current file
current_dir = os.path.dirname(os.path.abspath("__file__"))
# get the project root (one level up from notebooks/)
project_root = os.path.abspath(os.path.join(current_dir, ".."))
# add project root to sys.path
if project_root not in sys.path:
    sys.path.append(project_root)

In [42]:
%reload_ext autoreload
%autoreload 2

from utils.stock_utils import download_stock_data
from utils.kpi import CAGR, volatility, Sharpe, max_dd, Sortino, calamar

List of all the stocks I want to have in my tradable universe for backtesting

In [3]:
mid_cap = ["PAYTM.NS", "NYKAA.NS", "VMM.NS", "MFSL.NS", "EXIDEIND.NS", "UPL.NS", "BSE.NS", "POLICYBZR.NS", "M&MFIN.NS", "MANKIND.NS", "NATIONALUM.NS", "ABFRL.NS", "ESCORTS.NS", "WAAREEENER.NS", "TIINDIA.NS", "NMDC.NS", "KALYANKJIL.NS", "SUZLON.NS", "UNIONBANK.NS", "SRF.NS"]
# mid_cap = ["PAYTM.NS"]

# PORTFOLIO REBALANCING

In [45]:
mid_cap_dict = download_stock_data(mid_cap, 3650, "1mo")

Successfully downloaded 20/20 stocks.


In [ ]:
def get_monthly_return(mid_cap_dict):
    """
    Calculate monthly returns for the given mid-cap stocks.
    
    Parameters:
    mid_cap_dict (dictionary): Dictionary of stock symbols.
    
    Returns:
    pd.DataFrame: DataFrame containing monthly returns for each stock.
    """
    ohlcv_dict = copy.deepcopy(mid_cap_dict)
    return_df = pd.DataFrame()
    for stock in mid_cap:
        print("calculated monthly return for ", stock)
        ohlcv_dict[stock]["mon_ret"] = ohlcv_dict[stock]["Close"].pct_change()
        return_df[stock] = ohlcv_dict[stock]["mon_ret"]
    return_df.dropna(inplace=True)
    return return_df

In [47]:
return_df = get_monthly_return(mid_cap_dict)

calculated monthly return for  PAYTM.NS
calculated monthly return for  NYKAA.NS
calculated monthly return for  VMM.NS
calculated monthly return for  MFSL.NS
calculated monthly return for  EXIDEIND.NS
calculated monthly return for  UPL.NS
calculated monthly return for  BSE.NS
calculated monthly return for  POLICYBZR.NS
calculated monthly return for  M&MFIN.NS
calculated monthly return for  MANKIND.NS
calculated monthly return for  NATIONALUM.NS
calculated monthly return for  ABFRL.NS
calculated monthly return for  ESCORTS.NS
calculated monthly return for  WAAREEENER.NS
calculated monthly return for  TIINDIA.NS
calculated monthly return for  NMDC.NS
calculated monthly return for  KALYANKJIL.NS
calculated monthly return for  SUZLON.NS
calculated monthly return for  UNIONBANK.NS
calculated monthly return for  SRF.NS


In [48]:
def portfolio_rebalance(return_df, keep, throw):
    """
    Rebalance the portfolio based on monthly returns.
    
    Parameters:
    return_df (pd.DataFrame): DataFrame containing monthly returns for each stock.
    keep (int): Number of stocks to keep in the portfolio.
    throw (int): Number of stocks to throw out of the portfolio.
    initial_investment (float): Initial investment amount (default is 1,000,000).
    
    Returns:
    pd.DataFrame: DataFrame containing portfolio value over time.
    """
    df = return_df.copy()
    portfolio = []
    monthly_return = []
    for i in range (len(df)):
        if len(portfolio) > 0:
            monthly_return.append(df[portfolio].iloc[i, :].mean())
            bad_stocks = df[portfolio].iloc[i,:].sort_values(ascending=True)[:throw].index.values.tolist()
            portfolio = [t for t in portfolio if t not in bad_stocks]
        else:
            monthly_return.append(0)
            print("No stocks in portfolio")
        fill = keep - len(portfolio)
        new_picks = df.iloc[i,:].sort_values(ascending=False)[:fill].index.values.tolist()
        portfolio = portfolio + new_picks
        print(portfolio)
    monthly_ret_df = pd.DataFrame(np.array(monthly_return),columns=["mon_ret"])
    return monthly_ret_df

In [56]:
new_portfolio = portfolio_rebalance(return_df, 6, 3)
CAGR(new_portfolio, 12, "mon_ret", False)  # Calculate CAGR

No stocks in portfolio
['SRF.NS', 'UPL.NS', 'ESCORTS.NS', 'M&MFIN.NS', 'NYKAA.NS', 'VMM.NS']
['SRF.NS', 'UPL.NS', 'M&MFIN.NS', 'UPL.NS', 'SRF.NS', 'UNIONBANK.NS']
['SRF.NS', 'SRF.NS', 'UNIONBANK.NS', 'BSE.NS', 'MFSL.NS', 'SUZLON.NS']
['BSE.NS', 'MFSL.NS', 'BSE.NS', 'MFSL.NS', 'VMM.NS', 'KALYANKJIL.NS']
['BSE.NS', 'BSE.NS', 'SUZLON.NS', 'BSE.NS', 'UNIONBANK.NS', 'NATIONALUM.NS']
['UNIONBANK.NS', 'NATIONALUM.NS', 'SRF.NS', 'MFSL.NS', 'VMM.NS', 'NATIONALUM.NS']
['NATIONALUM.NS', 'VMM.NS', 'NATIONALUM.NS', 'PAYTM.NS', 'MANKIND.NS', 'KALYANKJIL.NS']
['VMM.NS', 'PAYTM.NS', 'WAAREEENER.NS', 'PAYTM.NS', 'NYKAA.NS', 'VMM.NS']
['VMM.NS', 'NYKAA.NS', 'VMM.NS', 'NATIONALUM.NS', 'NMDC.NS', 'UNIONBANK.NS']
['NYKAA.NS', 'NATIONALUM.NS', 'UNIONBANK.NS', 'BSE.NS', 'PAYTM.NS', 'M&MFIN.NS']
['NATIONALUM.NS', 'BSE.NS', 'M&MFIN.NS', 'M&MFIN.NS', 'BSE.NS', 'NATIONALUM.NS']
['NATIONALUM.NS', 'NATIONALUM.NS', 'NATIONALUM.NS', 'NMDC.NS', 'M&MFIN.NS', 'SRF.NS']
['NATIONALUM.NS', 'NATIONALUM.NS', 'NATIONALUM.NS'

np.float64(0.2783071077090564)

In [57]:
volatility(new_portfolio, 12, "mon_ret", False)  # Calculate Volatility

np.float64(0.22589922638084597)

In [58]:
Sharpe(new_portfolio, 12, "mon_ret", False)  # Calculate Sharpe Ratio

np.float64(1.099194148148311)

In [59]:
max_dd(new_portfolio, "mon_ret", False)

np.float64(0.09791643326593515)

In [ ]:
# 1. Download Data
ticker = "NIFTY_MIDCAP_100.NS"
data = yf.download(ticker, period="10y", interval="1mo")

# 2. Clean Data (yfinance multi-index handling if necessary)
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

# 3. Calculate KPIs
# Since we are using daily data, timeframe = 252
# Since we downloaded ohlcv, is_price = True
cagr_val = CAGR(data, timeframe=12, column='Close', is_price=True)
vol_val = volatility(data, timeframe=12, column='Close', is_price=True)
sharpe_val = Sharpe(data, timeframe=12, column='Close', is_price=True)
mdd_val = max_dd(data, column='Close', is_price=True)

# 4. Display Results
print(f"--- Statistics for {ticker} (Last 10 Years) ---")
print(f"CAGR:           {cagr_val:.2%}")
print(f"volatility:     {vol_val:.2%}")
print(f"Sharpe Ratio:   {sharpe_val:.2f}")
print(f"Max Drawdown:   {mdd_val:.2%}")

[*********************100%***********************]  1 of 1 completed

--- Statistics for NIFTY_MIDCAP_100.NS (Last 10 Years) ---
CAGR:           21.70%
volatility:     33.33%
Sharpe Ratio:   0.56
Max Drawdown:   44.62%
